# Integrating Tavily Search Tool into a LangChain Agent

## Overview

In this tutorial, you'll learn how to integrate the **Tavily Search Tool** into a LangChain prebuilt agent. By the end of this notebook, you'll have a working agent that can autonomously decide when to search the web for information to answer user queries.

## What You'll Learn

- How to set up and configure the Tavily search tool
- How to create a prebuilt agent that can use external tools
- How to observe the agent making decisions about when to use web search

## Prerequisites

- **API Keys Required**: You'll need both `OPENAI_API_KEY` and `TAVILY_API_KEY`
  - OpenAI: https://platform.openai.com/api-keys
  - Tavily: https://tavily.com (free account available)
- **Required Packages**: `langchain`, `langchain-openai`, `langchain-tavily`, `python-dotenv`

## Setup Instructions

1. Create a `.env` file in your project directory
2. Add your API keys:
   ```
   OPENAI_API_KEY=your-openai-key-here
   TAVILY_API_KEY=your-tavily-key-here
   ```
3. Install required packages:
   ```bash
   pip install langchain langchain-openai langchain-tavily python-dotenv
   ```

## Step 1: Import Libraries and Load Environment Variables

First, we'll import all necessary libraries and load our API keys from the `.env` file using `python-dotenv`.

In [5]:
import os
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

# Load environment variables from .env file
load_dotenv()

# Verify API keys are loaded
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found in environment variables")
if not os.getenv("TAVILY_API_KEY"):
    raise ValueError("TAVILY_API_KEY not found in environment variables")

print("API keys loaded successfully!")

API keys loaded successfully!


## Step 2: Configure the Language Model

We'll create an instance of ChatOpenAI that will power our agent's reasoning capabilities. The agent uses this model to understand queries and decide when to use tools.

In [6]:
# Configure the model with a low temperature for more consistent reasoning
model = ChatOpenAI(
    model="gpt-5",  # Using GPT-5 for reliable tool usage
    temperature=0.1   # Low temperature for more deterministic responses
)

print(f"Model configured: {model.model_name}")

Model configured: gpt-5


## Step 3: Create the Tavily Search Tool

The Tavily search tool provides real-time web search capabilities. We'll configure it with parameters that control how searches are performed:

- **max_results**: Number of search results to return (default: 5)
- **search_depth**: "basic" for faster searches or "advanced" for more comprehensive results
- **include_raw_content**: Whether to include full page content (we'll keep this False for cleaner results)
- **include_images**: Whether to include images in results

In [7]:
# Instantiate the Tavily search tool
search_tool = TavilySearch(
    max_results=5,
    search_depth="basic",  # Use "advanced" for more comprehensive results
    include_raw_content=False,
    include_images=False
)

print("Tavily search tool created successfully!")
print(f"Tool name: {search_tool.name}")
print(f"Tool description: {search_tool.description}")

Tavily search tool created successfully!
Tool name: tavily_search
Tool description: A search engine optimized for comprehensive, accurate, and trusted results. Useful for when you need to answer questions about current events. It not only retrieves URLs and snippets, but offers advanced search depths, domain management, time range filters, and image search, this tool delivers real-time, accurate, and citation-backed results.Input should be a search query.


## Step 4: Create the Agent with Tavily Tool

Now we'll create a prebuilt agent using LangChain's `create_agent` function. This agent will:

1. Receive a user query
2. Reason about whether it needs to use the search tool
3. If needed, invoke the Tavily search tool
4. Synthesize the search results into a coherent answer

The agent automatically decides when to use the search tool based on the query.

In [8]:
# Create the agent with the search tool
agent = create_agent(
    model=model,
    tools=[search_tool]  # Pass our Tavily search tool in a list
)

print("Agent created successfully!")
print(f"Agent has access to {len([search_tool])} tool(s)")

Agent created successfully!
Agent has access to 1 tool(s)


## Step 5: Create a Helper Function for Agent Interactions

To make it easier to interact with our agent, we'll create a helper function that:

- Takes a query string
- Invokes the agent with properly formatted messages
- Extracts and prints the final response

This function simplifies our testing and makes the examples more readable.

In [9]:
def generate_and_print_response(agent, query):
    """
    Invoke the agent with a query and print the response.
    
    Args:
        agent: The LangChain agent instance
        query: The user query string
    """
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}\n")
    
    # Invoke the agent with the query
    result = agent.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    
    # Extract and print the final response
    final_message = result["messages"][-1]
    print(f"Response: {final_message.content}\n")
    
    return result

print("Helper function defined successfully!")

Helper function defined successfully!


## Step 6: Test the Agent - Current Events Query

Let's test our agent with a query about recent developments. The agent should recognize that it needs current information and automatically use the Tavily search tool.

In [10]:
# Example 1: Current events query
result1 = generate_and_print_response(
    agent,
    "What are the latest developments in artificial intelligence in 2025?"
)


Query: What are the latest developments in artificial intelligence in 2025?

Response: Here’s a concise rundown of notable AI developments in 2025, with sources for each item:

- Frontier models and multimodality
  - OpenAI: Released GPT-4.5 (Feb), new image-generation API model (Apr), published Sora 2 video model research (Sep), and updated the GPT-5 system card with safety addendum (Oct) (openai.com/news; GPT-4.5: https://openai.com/index/introducing-gpt-4-5/; Image API: https://openai.com/index/image-generation-api/; Sora 2: https://openai.com/index/sora-2/; GPT-5 addendum: https://openai.com/index/gpt-5-system-card-sensitive-conversations/).
  - Google: Upgraded Gemini 2.5 Pro and introduced “Deep Think” reasoning mode; Veo 3.1 and 3.1 Fast video models entered public preview via the Gemini API; rolled out “AI Mode” in Search to US users (Google blog: https://blog.google/technology/google-deepmind/google-gemini-updates-io-2025/; Gemini API changelog: https://ai.google.dev/gemini-a

## Step 7: Test the Agent - Specific Topic Search

Now let's try a more specific query about a particular technology or company. The agent should use Tavily to find up-to-date information.

In [11]:
# Example 2: Specific topic search
result2 = generate_and_print_response(
    agent,
    "What are the key features of LangChain's newest releases?"
)


Query: What are the key features of LangChain's newest releases?

Response: Here’s a concise roundup of the most recent LangChain/LangGraph releases and what they add:

Core LangChain v1 (Python and JS)
- New standard agent API: create_agent / createAgent replaces older patterns, making agent building simpler and more consistent across providers and platforms [Docs v1 (Python, JS), v1 migration guides].
- Built-in middleware: first-class patterns for PII redaction, summarization/auto-condense, and human-in-the-loop approvals/intercepts; replaces pre/post model hooks [Docs v1].
- Standard “content blocks”: a unified, multimodal message format used across models and tools for more reliable interactions [v1 announcement].
- Updated semantics and migration tweaks: system_prompt naming, structured output now via Tool/Provider strategies (prompted structured output removed), updated return types, and streaming node rename [v1 migration guides].
- New docs site and migration guides to v1 [v1